### Import dependencies

In [5]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
import os

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig

from google.genai import types

from dotenv import load_dotenv

load_dotenv("../../.env")

from utils.tools_2 import check_warehouse_availability, reserve_warehouse_items

### ADK Agent

In [6]:
model = LiteLlm(
    model="openai/gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [7]:
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    model=model,
    tools=[
        check_warehouse_availability,
        reserve_warehouse_items
    ],
    description="The user is asking to reserve items from the warehouses or about availability of the items in warehouses.",
    instruction="""You are a part of the shopping assistant that can manage available inventory in the warehouses.

## Instructions

- As the final answer you should return an answer to the users query in a form of actions performed.
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combine the required quantity from different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.
- As the final answer you should return an answer in a form of actions performed.
"""
)

### ADK Session

In [8]:
session_service = InMemorySessionService()

In [9]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="sessions_1"
)

Session(id='sessions_1', app_name='warehouse_app', user_id='user_1', state={}, events=[], last_update_time=1786064262.819978)

In [10]:
runner = Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app"
)

In [15]:
message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="What is the availability of B0BRJS644Z in all of your warehouses?"
        )
    ]
)

In [16]:
result = runner.run(
    user_id="user_1",
    session_id="sessions_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=5
    )
)  

In [17]:
result

<generator object Runner.run at 0x1170fd5a0>

In [14]:
for event in result:
    print("======================")
    print(event)

/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


model_version='gpt-5.4-mini-2026-03-17' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_HSA2wrIZHnSJ8TLmA0S3Dwpj',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=36,
  prompt_token_count=643,
  total_token_count=679
) live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocation_id='e-db3e81b6-754a-4

### Agent Run Wrapper

In [18]:
async def run_warehouse_agent(query: str, session_id: str, session_service: InMemorySessionService) -> str:

    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id="user_1",
            session_id=session_id
        )

    runner = Runner(
        agent=warehouse_agent,
        session_service=session_service,
        app_name="warehouse_app"
    )

    content = types.Content(
        role="user",
        parts=[
            types.Part(
                text=query
            )
        ]
    )

    final_text = ""
    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=5
        )
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_text += part.text

    return final_text

In [20]:
answer_1 = await run_warehouse_agent(
    query="What is the availability of B0BRJS644Z in all of your warehouses?",
    session_id="test_01",
    session_service=session_service
)

In [21]:
print(answer_1)

Availability check completed for product **B0BRJS644Z**:

- **Berlin Distribution Center (DE-BER-01):** 73 available
- **Lyon Regional Warehouse (FR-LYO-01):** 0 available
- **Munich Logistics Hub (DE-MUN-01):** 27 available
- **Paris Central Depot (FR-PAR-01):** 6 available
- **Marseille Mediterranean Hub (FR-MAR-01):** 4 available
- **Hamburg North Warehouse (DE-HAM-01):** 9 available

**Result:** The item is **available in 5 of 6 warehouses** and can be fulfilled completely from any of those warehouses.


In [22]:
answer_2 = await run_warehouse_agent(
    query="Can you reserve 9 of this in the Munich warehouse?",
    session_id="test_01",
    session_service=session_service
)

In [ ]:
print(answer_2)

Reservation completed successfully.

Actions performed:
- Reserved **9 units** of **B0BRJS644Z**
- Warehouse: **Munich Logistics Hub (DE-MUN-01), Munich, Germany**

All requested items were reserved successfully.
